# Session 10 · Pig Latin Data Model and Operators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-10-pig-latin-data-model-and-operators.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Tuples, bags, GROUP, COGROUP and FLATTEN in Python

In [2]:
from collections import defaultdict
students = [("Ali", "Karachi", 82), ("Sara", "Lahore", 91), ("Omar", "Karachi", 77),
            ("Hina", "Lahore", None), ("Zain", "Karachi", 95)]
fees = [("Karachi", 45000), ("Lahore", 42000), ("Quetta", 30000)]

def fmt_bag(bag):
    return "{" + ", ".join("(" + ",".join("" if v is None else str(v) for v in t) + ")" for t in bag) + "}"

group = defaultdict(list)                                         # GROUP students BY city
for t in students:
    group[t[1]].append(t)
print("GROUP students BY city:")
for k, bag in group.items():
    print(f"  ({k}, {fmt_bag(bag)})")

print("\nFOREACH g GENERATE group, COUNT(students.score), AVG(students.score):")
for k, bag in group.items():
    scores = [t[2] for t in bag if t[2] is not None]             # nulls are skipped
    print(f"  ({k}, {len(scores)}, {sum(scores)/len(scores):.1f})")

print("\nCOGROUP students BY city, fees BY city:")
keys = sorted({t[1] for t in students} | {f[0] for f in fees})
for k in keys:
    print(f"  ({k}, {fmt_bag([t for t in students if t[1] == k])}, {fmt_bag([f for f in fees if f[0] == k])})")

print("\nFLATTEN back to flat tuples:")
for k, bag in group.items():
    for t in bag:
        print(f"  ({k},{t[0]})")

GROUP students BY city:
  (Karachi, {(Ali,Karachi,82), (Omar,Karachi,77), (Zain,Karachi,95)})
  (Lahore, {(Sara,Lahore,91), (Hina,Lahore,)})

FOREACH g GENERATE group, COUNT(students.score), AVG(students.score):
  (Karachi, 3, 84.7)
  (Lahore, 1, 91.0)

COGROUP students BY city, fees BY city:
  (Karachi, {(Ali,Karachi,82), (Omar,Karachi,77), (Zain,Karachi,95)}, {(Karachi,45000)})
  (Lahore, {(Sara,Lahore,91), (Hina,Lahore,)}, {(Lahore,42000)})
  (Quetta, {}, {(Quetta,30000)})

FLATTEN back to flat tuples:
  (Karachi,Ali)
  (Karachi,Omar)
  (Karachi,Zain)
  (Lahore,Sara)
  (Lahore,Hina)


### Example 3 · The same analysis, run for real in PySpark

In [3]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("pig-logs")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

logs = spark.createDataFrame([
    ("10.0.0.1", "10:01", "/home", 200, 5120), ("10.0.0.2", "10:02", "/admissions", 200, 8192),
    ("10.0.0.1", "10:03", "/results", 500, 0), ("10.0.0.3", "10:04", "/home", 200, 5120),
    ("10.0.0.1", "10:05", "/results", 500, 0), ("10.0.0.4", "10:06", "/fees", 404, 0),
    ("10.0.0.2", "10:07", "/fees", 200, 3072), ("10.0.0.1", "10:08", "/admissions", 503, 0),
    ("10.0.0.3", "10:09", "/results", 200, 9216), ("10.0.0.2", "10:10", "/home", 200, 5120)],
    "ip string, ts string, url string, status int, bytes long")
pages = spark.createDataFrame([("/home", "main"), ("/admissions", "students"), ("/results", "students"),
                               ("/fees", "finance")], "url string, section string")

errors = logs.filter("status >= 500")                     # SPLIT ... errors IF status >= 500
ok = logs.filter("status < 400")                          #           ok     IF status < 400
print("errors by url:")
errors.groupBy("url").agg(F.count("*").alias("n")).orderBy(F.desc("n")).show()

print("traffic by section (replicated join = broadcast):")
(ok.join(F.broadcast(pages), "url")
   .groupBy("section").agg(F.count("*").alias("hits"), F.round(F.sum("bytes") / 1024.0, 1).alias("kb"))
   .orderBy(F.desc("hits"), "section").show())

print("top 3 IPs:")
logs.groupBy("ip").agg(F.count("*").alias("n")).orderBy(F.desc("n"), "ip").limit(3).show()

print("Pig-style bag per IP (collect_list = bag):")
logs.groupBy("ip").agg(F.collect_list(F.struct("url", "status")).alias("bag")).orderBy("ip").show(truncate=False)

errors by url:


+-----------+---+
|        url|  n|
+-----------+---+
|   /results|  2|
|/admissions|  1|
+-----------+---+

traffic by section (replicated join = broadcast):


+--------+----+----+
| section|hits|  kb|
+--------+----+----+
|    main|   3|15.0|
|students|   2|17.0|
| finance|   1| 3.0|
+--------+----+----+

top 3 IPs:


+--------+---+
|      ip|  n|
+--------+---+
|10.0.0.1|  4|
|10.0.0.2|  3|
|10.0.0.3|  2|
+--------+---+

Pig-style bag per IP (collect_list = bag):


+--------+--------------------------------------------------------------------+
|ip      |bag                                                                 |
+--------+--------------------------------------------------------------------+
|10.0.0.1|[{/home, 200}, {/results, 500}, {/results, 500}, {/admissions, 503}]|
|10.0.0.2|[{/admissions, 200}, {/fees, 200}, {/home, 200}]                    |
|10.0.0.3|[{/home, 200}, {/results, 200}]                                     |
|10.0.0.4|[{/fees, 404}]                                                      |
+--------+--------------------------------------------------------------------+



### Example 4 · Nested FOREACH: top 2 per group

In [4]:
from pyspark.sql import SparkSession, functions as F, Window
spark = (SparkSession.builder.master("local[2]").appName("nested")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
students = spark.createDataFrame([("Ali", "Karachi", 82), ("Sara", "Lahore", 91), ("Omar", "Karachi", 77),
                                  ("Hina", "Lahore", 88), ("Zain", "Karachi", 95), ("Iqra", "Lahore", 73)],
                                 "name string, city string, score int")
w = Window.partitionBy("city").orderBy(F.desc("score"))
(students.withColumn("pos", F.row_number().over(w)).filter("pos <= 2")    # ORDER + LIMIT inside each bag
         .groupBy("city").agg(F.collect_list("name").alias("top2")).orderBy("city").show())

+-------+------------+
|   city|        top2|
+-------+------------+
|Karachi| [Zain, Ali]|
| Lahore|[Sara, Hina]|
+-------+------------+



### Example 5 · Nulls, casting and DISTINCT

In [5]:
rows = [("Ali", "82"), ("Sara", "91"), ("Omar", ""), ("Hina", "abc"), ("Sara", "91")]

def to_int(b):                       # Pig: (int)$1 gives null when the bytearray is not a number
    try:
        return int(b)
    except ValueError:
        return None

typed = [(n, to_int(s)) for n, s in rows]
print("after cast       :", typed)
print("FILTER score > 80:", [t for t in typed if t[1] is not None and t[1] > 80])
print("COUNT(score)     :", sum(t[1] is not None for t in typed), "| COUNT_STAR:", len(typed))
print("DISTINCT         :", sorted(set(typed), key=str))

after cast       : [('Ali', 82), ('Sara', 91), ('Omar', None), ('Hina', None), ('Sara', 91)]
FILTER score > 80: [('Ali', 82), ('Sara', 91), ('Sara', 91)]
COUNT(score)     : 3 | COUNT_STAR: 5
DISTINCT         : [('Ali', 82), ('Hina', None), ('Omar', None), ('Sara', 91)]
